# Graph-structured predictive coding — a walkthrough

## What you'll build

Your current `pc_infer` handles a **chain**: every node has exactly one child, so
Equation 2's sum over children has exactly one term.

$$\frac{dv_i}{dt} = -\frac{\partial \mathcal{F}}{\partial v_i}, \qquad
\frac{\partial \mathcal{F}}{\partial v_i} = \epsilon_i - \sum_{j \in C(v_i)} \epsilon_j \frac{\partial \hat v_j}{\partial v_i}$$

A ViT block is not a chain. A residual connection means one node's value feeds two
downstream nodes, so that sum genuinely has multiple terms. You need that case working
and checked against autodiff before a PC predictor can go inside a JEPA stack.

By the end you will have written, from the equations:

1. `infer` — Equation 2 for an arbitrary DAG, with a real multi-term children sum
2. `pc_weight_grads` — Equation 3 for arbitrary edge functions
3. the measurement that shows what inference budget $T$ a branching graph needs

## How to use this notebook

Cells marked **`# TODO`** are yours to write. Each one is followed by a **Check** cell
that prints something, and the markdown immediately after tells you the number you should
see. If it doesn't match, the bug is in the cell you just wrote — not later.

Scaffolding you don't need to derive (the autodiff reference, the plotting) is given to
you complete.

Reference throughout: Millidge, Tschantz & Buckley, *Predictive Coding Approximates
Backprop along Arbitrary Computation Graphs* (arXiv:2006.04182v5). Section and appendix
pointers are given per step.

**If you get stuck**, a verified reference implementation sits next to this notebook in
`graph_pc.py`. Try each exercise first — the checks are there so you can find your own
bug, which is where the understanding is.

**Runtime**: CPU only, ~5 minutes total, nearly all of it in the training section.

## 0. Setup

In [6]:
import jax, jax.numpy as jnp, jax.random as jr
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

jax.config.update("jax_platform_name", "cpu")   # all small; avoids GPU init noise
print("jax", jax.__version__, "| numpy", np.__version__)

jax 0.11.0 | numpy 2.5.1


## 1. Representing a computation graph

The paper works with an *augmented computation graph* $\tilde{\mathcal{G}}$ (§2, p. 3):
vertices $v_i$, each with a parent set $P(v_i)$ and an edge function
$\hat v_i = f(P(v_i); \theta_i)$ producing its prediction.

We encode exactly that. A graph is a list where entry $i$ is `(parents, fn)`:

- `parents` — list of node indices feeding node $i$
- `fn(parent_values, theta_i)` — computes $\hat v_i$

Conventions: node `0` is the input (clamped to the data, no parents, no prediction); the
**last** node is the output (clamped to the target during learning); everything between
is free to relax.

Below is the paper's Figure 2 test function (p. 7),
$v_4 = \tan\!\big(\sqrt{\theta v_0}\big) + \sin(v_0^2)$, written as a graph. Read it
carefully — note that $v_0$ appears in **two** places.

In [7]:
theta_val = jnp.array([2.0])

fig2_nodes = [
    ([],     None),                                            # v0 : input (clamped)
    ([0],    lambda p, th: p[0] * th),                         # v1 = theta * v0
    ([1],    lambda p, th: jnp.sqrt(p[0])),                    # v2 = sqrt(v1)
    ([0],    lambda p, th: p[0] ** 2),                         # v3 = v0^2     <- second use of v0
    ([2, 3], lambda p, th: jnp.tan(p[0]) + jnp.sin(p[1])),     # v4 = tan(v2) + sin(v3)
]
fig2_thetas = [None, theta_val, None, None, None]   # only edge 1 is parameterised

x      = jnp.array([5.0])
target = jnp.array([1.0])

### Exercise 1 — the children map

Equation 2 needs $C(v_i)$, the children of node $i$. That's the *reverse* of the parent
lists, and you have to build it.

One subtlety that will bite you if you skip it: a child with two parents has a
**different Jacobian with respect to each one**. So it isn't enough to know that node $j$
is a child of node $i$ — you need to know **which parent slot** $i$ occupies in $j$.

Return a list where `children[i]` is a list of `(j, slot)` pairs meaning "node $j$ has
node $i$ as its `slot`-th parent".

In [ ]:
def children_of(nodes):
    ch = [[] for _ in nodes]
    for j, (parents, _) in enumerate(nodes):
        for slot, p in enumerate(parents):
            ch[p].append((j, slot))
    return ch

In [9]:
# Check
ch = children_of(fig2_nodes)
for i, c in enumerate(ch):
    print(f"children of node {i}: {c}")

children of node 0: []
children of node 1: []
children of node 2: []
children of node 3: []
children of node 4: []


**Check:** you should see

```
children of node 0: [(1, 0), (3, 0)]
children of node 1: [(2, 0)]
children of node 2: [(4, 0)]
children of node 3: [(4, 1)]
children of node 4: []
```

Two things to read off this.

**Node 0 has two children.** Its update will have two terms in the summation. This is the
branch point a chain implementation cannot express.

**Node 3 sits in slot 1 of node 4**, while node 2 sits in slot 0. Node 4 computes
$\tan(v_2) + \sin(v_3)$ — the Jacobian with respect to slot 0 is $\sec^2$, with respect to
slot 1 it's $\cos$. Getting the slot wrong silently swaps two derivatives.

**Node 4 has none.** With no children the sum in Equation 2 vanishes and the update
reduces to $dv_i/dt = -\epsilon_i$, so the node would relax onto its own prediction and
give $\epsilon_i = 0$ — no learning signal reaching anything upstream. That is exactly
why the output node is **clamped** to the target rather than left free.

### Exercise 2 — the forward pass

Algorithm 1's first loop (§2, p. 5): visit nodes in order, compute each prediction from
its parents. Because children always have higher indices than their parents here, one
left-to-right sweep suffices.

Return the list of all node values, with `v[0] = x`.

In [ ]:
def forward(nodes, thetas, x):
    v = [x] + [None] * (len(nodes) - 1)
    for i in range(1, len(nodes)):
        parents, fn = nodes[i]
        v[i] = fn([v[p] for p in parents], thetas[i])
    return v

def loss_fn(nodes, thetas, x, target):
    v = forward(nodes, thetas, x)
    return 0.5 * jnp.sum((target - v[-1]) ** 2)

In [11]:
# Check
v_ff = forward(fig2_nodes, fig2_thetas, x)
print("v0..v4:", [round(float(a.ravel()[0]), 6) for a in v_ff])
print("loss  :", round(float(loss_fn(fig2_nodes, fig2_thetas, x, target)), 6))

AttributeError: 'NoneType' object has no attribute 'ravel'

**Check:**

```
v0..v4: [5.0, 10.0, 3.162278, 25.0, -0.111664]
loss  : 0.617898
```

### Given: the backprop reference

You need $\partial L/\partial v_i$ to check PC against, and you should not hand-derive it.
This is given complete — for each node $i$ it rebuilds the graph *from node $i$ forward*
as a function of $v_i$, then calls `jax.grad`. Nothing PC-specific here; it's the ground
truth.

In [ ]:
def true_node_grads(nodes, thetas, x, target):
    v_ff = forward(nodes, thetas, x)
    grads = []
    for i in range(len(nodes)):
        def L_from_i(vi, i=i):
            v = list(v_ff)
            v[i] = vi
            for j in range(i + 1, len(nodes)):
                parents, fn = nodes[j]
                v[j] = fn([v[p] for p in parents], thetas[j])
            return 0.5 * jnp.sum((target - v[-1]) ** 2)
        grads.append(jax.grad(L_from_i)(v_ff[i]))
    return grads

tg = true_node_grads(fig2_nodes, fig2_thetas, x, target)
print("backprop dL/dv_i:", [round(float(g.ravel()[0]), 6) for g in tg])

This prints

```
backprop dL/dv_i: [-11.370533, -0.175845, -1.11214, -1.101884, -1.111664]
```

The gradient at the branch node, $-11.370533$, is the one to watch. It is the **sum of two
paths** — through $v_1$ and through $v_3$. If your multi-child summation is wrong, this is
the number that will be wrong, and it's the only one that tests branching at all.

### Given: the Jacobian-transpose helper

The term $\epsilon_j \frac{\partial \hat v_j}{\partial v_i}$ is a vector-Jacobian
product: take the child's error and pull it back through the child's edge function.
`jax.vjp` does this for any `fn` you write, so unlike the paper's reference
implementation you never hand-derive a derivative.

The one thing to notice: when the child has several parents we hold the others fixed and
differentiate only with respect to slot `slot`. That's what the `vp if k == slot else ...`
does.

In [ ]:
def jT_factory(nodes, thetas):
    '''Returns jT(j, slot, parent_vals, e) = (d fn_j / d parent[slot])^T @ e.'''
    def jT(j, slot, parent_vals, e):
        f = lambda vp: nodes[j][1](
            [vp if k == slot else parent_vals[k] for k in range(len(parent_vals))],
            thetas[j])
        _, vjp = jax.vjp(f, parent_vals[slot])
        return vjp(e)[0]
    return jT

## 2. Exercise 3 — Equation 2 for an arbitrary graph

This is the core of the notebook. Three things to get right, and each one is a decision
you have to make rather than a line to transcribe.

**The sum.** For each free node $i$, accumulate one `jT(...)` term **per child**. On a
chain that loop runs once; on a branch point it runs twice. Writing the loop is the whole
exercise.

**The sign.** Main text Equation 2 (p. 4) and Appendix D p. 24 disagree on the overall
sign, and Algorithm 1 on p. 5 prints `+`, which would be gradient *ascent*. Appendix D is
the correct one: differentiating $\mathcal{F} = \sum_i \epsilon_i^T \epsilon_i$ gives

$$\frac{\partial \mathcal{F}}{\partial v_i} = \epsilon_i - \sum_{j \in C(v_i)} \epsilon_j \frac{\partial \hat v_j}{\partial v_i}$$

and the dynamics are the **negative** of that.

Be careful here, because it is easy to negate twice. Put $\partial \mathcal{F}/\partial v_i$
itself in the variable — the expression above, `eps[i] - child_term`, *not* its negative —
and then subtract it times the step size. If you store $-\partial \mathcal{F}/\partial v_i$
and also subtract, the two cancel and you get gradient **ascent**, which does not error and
does not obviously look wrong for a few hundred steps. Exercise 3.5 below is there to catch
exactly that.

(You resolved the sign itself earlier with the childless-node case: $dv/dt = -\epsilon$
relaxes onto the prediction, $+\epsilon$ diverges.)

**The fixed-prediction assumption** (§2, p. 5). The paper freezes the predictions
$\hat v_i$ at their feedforward values throughout the relaxation, severing the coupling
between a parent's activity and its child's prediction — which is what makes each node's
problem local. Implement it as a flag, because you're going to measure what it buys and
it is not a detail.

Concretely, the flag controls **where the Jacobians are evaluated**: at the frozen
feedforward parent values (`True`), or at the current relaxed ones (`False`). With
`False` you must also recompute all `mu` at the end of each step.

Integration is forward Euler, as in the paper:
$v_i^{t+1} \leftarrow v_i^t - \eta_v \frac{\partial \mathcal{F}}{\partial v_i^t}$.

**Which nodes move:** only `1 .. N-1`. Node `0` is clamped to the data and node `N` to
the target.

In [ ]:
def infer(nodes, thetas, x, target, n_steps, lr, fixed_prediction=True):
    '''Relax the free vertices under Eq. 2.

    Returns (v, mu, eps_hist) where eps_hist[t][i] is epsilon_i BEFORE step t --
    so entry 0 is the feedforward state and entry n_steps is the final state.
    '''
    N   = len(nodes) - 1
    ch  = children_of(nodes)
    jT  = jT_factory(nodes, thetas)

    v_ff = forward(nodes, thetas, x)
    mu   = list(v_ff)          # predictions
    v    = list(v_ff)
    v[N] = target              # clamp the output node

    eps_hist = []
    for t in range(n_steps):
        eps = [jnp.zeros_like(v[0])] + [v[i] - mu[i] for i in range(1, N + 1)]
        eps_hist.append([jnp.asarray(e) for e in eps])

        base  = v_ff if fixed_prediction else v   # where Jacobians are evaluated
        new_v = list(v)

        for i in range(1, N):                     # node 0 and node N are clamped
            child_term = jnp.zeros_like(v[i])
            for (j, slot) in ch[i]:               # one term PER CHILD -- the whole point
                pv = [base[p] for p in nodes[j][0]]
                child_term = child_term + jT(j, slot, pv, eps[j])

            # Appendix D:  dF/dv_i = eps_i - sum_j J_j^T eps_j
            # Store dF/dv itself, then SUBTRACT it. Storing -dF/dv and also
            # subtracting negates twice and gives gradient ASCENT.
            dF_dvi   = eps[i] - child_term
            new_v[i] = v[i] - lr * dF_dvi         # forward Euler descent

        v = new_v

        if not fixed_prediction:
            for i in range(1, N + 1):
                parents, fn = nodes[i]
                mu[i] = fn([v[p] for p in parents], thetas[i])

    eps = [jnp.zeros_like(v[0])] + [v[i] - mu[i] for i in range(1, N + 1)]
    eps_hist.append([jnp.asarray(e) for e in eps])
    return v, mu, eps_hist

### Exercise 3.5 - check that your relaxation descends

Before looking at any gradient, verify the one property that makes `infer` a relaxation at
all: the energy $\mathcal{F} = \frac12 \sum_{i \geq 1} \|\epsilon_i\|^2$ must **settle**.
A sign error turns the update into gradient ascent, which produces finite,
plausible-looking numbers for hundreds of steps and then quietly diverges.

Run the cell below. You do not need to write anything.

In [ ]:
def energy(eps):
    return 0.5 * sum(float(jnp.sum(e ** 2)) for e in eps[1:])

_, _, h = infer(fig2_nodes, fig2_thetas, x, target, n_steps=800, lr=0.02)

print("F over the relaxation:")
for t in [0, 5, 20, 100, 400, 800]:
    print(f"   t={t:<5} F = {energy(h[t]):.6e}")

print()
print("eps_2 settling (increments should SHRINK):")
print("  ", [round(float(h[t][2].ravel()[0]), 6) for t in range(0, 61, 10)])

**Check:**

```
F over the relaxation:
   t=0     F = 6.178982e-01
   t=5     F = 6.292115e-01
   t=20    F = 7.533537e-01
   t=100   F = 1.545409e+00
   t=400   F = 1.858018e+00
   t=800   F = 1.858791e+00

eps_2 settling (increments should SHRINK):
   [0.0, 0.203441, 0.369666, 0.505485, 0.616458, 0.707132, 0.781219]
```

$\mathcal{F}$ rises from the feedforward state - expected, because clamping the output to
the target *injects* error at node $N$ - then flattens at $\approx 1.8588$ and stays there.
That plateau is the fixed point. The `eps_2` increments shrink each step, which is what
convergence looks like.

**If instead you see** $\mathcal{F}$ reaching $\sim\!10^{14}$ by $t=800$, with `eps_2`
increments *growing* (`[0.0, -0.243552, -0.540441, -0.902347, ...]`), you are ascending.
The cause is almost always a double negation: storing $-\partial\mathcal{F}/\partial v_i$
in `dF_dvi` and *also* subtracting it. One or the other, not both.

Everything downstream of `infer` reads its $\epsilon$, so a sign error here makes
Exercises 4-7 look broken while they are in fact correct. Get this plateau first.

### Exercise 4 — the gradient at the input node

The input is clamped so it never relaxes, but the paper's Figure 2 tracks exactly this
quantity, and it's the only number that tests the branch. It's the sum over node 0's
children:

$$\text{PC gradient at } v_0 = -\sum_{j \in C(v_0)} \epsilon_j \frac{\partial \hat v_j}{\partial v_0}$$

Note the leading minus: $\epsilon$ approximates $-\partial L/\partial v$ in our sign
convention, so the pulled-back sum needs negating to be comparable with `jax.grad`.

In [ ]:
def pc_input_grad(nodes, thetas, x, eps, v_ff):
    jT  = jT_factory(nodes, thetas)
    tot = jnp.zeros_like(x)
    for (j, slot) in children_of(nodes)[0]:
        pv  = [v_ff[p] for p in nodes[j][0]]
        tot = tot + jT(j, slot, pv, eps[j])
    return -tot

## 3. Testing the paper's central claim

§2.1 (pp. 5–6) proves that at the equilibrium of the inference dynamics
$\epsilon_i^* = \partial L/\partial v_i$ — with our sign convention, $-\epsilon_i^*$.
That is the claim the whole paper rests on.

Run it on the branching function, both with and without the fixed-prediction assumption.
**Predict before you run:** do you expect the assumption to matter a little or a lot?

In [ ]:
for label, fp in [("fixed-prediction (the paper's assumption)", True),
                  ("predictions recomputed each step",          False)]:
    v, mu, hist = infer(fig2_nodes, fig2_thetas, x, target,
                        n_steps=800, lr=0.02, fixed_prediction=fp)
    eps = hist[-1]
    g0  = pc_input_grad(fig2_nodes, fig2_thetas, x, eps, v_ff)
    print(f"\n{label}")
    print("  -eps_i at equilibrium :", [round(-float(e.ravel()[0]), 6) for e in eps[1:]])
    print("  backprop dL/dv_i      :", [round(float(g.ravel()[0]), 6) for g in tg[1:]])
    print(f"  PC grad at branch v0  : {float(g0.ravel()[0]):.6f}"
          f"   backprop: {float(tg[0].ravel()[0]):.6f}")
    print(f"  abs error at v0       : {abs(float(g0.ravel()[0]) - float(tg[0].ravel()[0])):.3e}")

**Check:**

```
fixed-prediction (the paper's assumption)
  -eps_i at equilibrium : [-0.175819, -1.112128, -1.101837, -1.111664]
  backprop dL/dv_i      : [-0.175845, -1.11214, -1.101884, -1.111664]
  PC grad at branch v0  : -11.370010   backprop: -11.370533
  abs error at v0       : 5.226e-04

predictions recomputed each step
  -eps_i at equilibrium : [-0.064441, -0.409017, -0.328506, -0.334856]
  backprop dL/dv_i      : [-0.175845, -1.11214, -1.101884, -1.111664]
  PC grad at branch v0  : -3.413946   backprop: -11.370533
  abs error at v0       : 7.957e+00
```

If your top block matches, your children sum is correct — including at the two-child node.
The $5\times10^{-4}$ residual is Euler integration error and shrinks with more steps.

**If only the branch node $v_0$ is wrong** while $v_1 \dots v_4$ are right, you have a
single-child bug: probably summing only the first child, or overwriting `child_term`
instead of accumulating.

**If node 3 is right but node 2 is wrong (or vice versa)**, you swapped a slot.

**If the magnitudes are enormous** ($10^6$ or larger), or `fixed_prediction=False` gives
`nan`, your update is ascending rather than descending — see Exercise 3.5. Note that the
*last* entry (`-1.111664`) will still be correct even then: node $N$ is clamped and its
prediction is frozen, so $\epsilon_N$ never changes during the relaxation. It matching is
not evidence that anything else is right.

**Now read the second block.** Recomputing the predictions gives $-3.41$ against a true
$-11.37$ — off by a factor of 3.3, and it is not converging to the backprop gradient at
all. Every $\epsilon_i$ is roughly a third of its correct value.

So the exact-equivalence result **depends on freezing the predictions**. Without it you
are minimising a different (arguably more principled) objective with a different fixed
point. This matters for your proposal: the assumption comes from Whittington & Bogacz, not
from the variational derivation in Appendix D, and it is the step a reviewer would
question. The second block is what it costs to drop it.

## 4. Exercise 5 — three topologies

Now build the structures that matter for a predictor. All three have the same number of
parameterised edges, so any difference you measure is topology, not capacity.

| name | recursion | why it's here |
|---|---|---|
| `mk_chain` | $v_i = \tanh(W_i v_{i-1})$ | your existing case, the control |
| `mk_resid_folded` | $v_i = \tanh(W_i v_{i-1}) + v_{i-1}$ | skip written **inside** one edge function |
| `mk_resid_explicit` | $a_i = \tanh(W_i v_{i-1})$, then $v_i = v_{i-1} + a_i$ | skip as a **separate merge node** — the ViT form |

`mk_chain` is given as the pattern. The other two are yours.

The difference between the last two looks like bookkeeping and is not — that is the point
of building both. In `resid_folded` the graph is still a chain: node $i-1$ has one child
and the identity path lives *inside* that child's Jacobian. In `resid_explicit` node $i-1$
genuinely has **two** children, which is how a transformer block is actually structured
(`x + Attn(x)`, then `x + MLP(x)`).

For `mk_resid_explicit`, each block appends **two** nodes: a sublayer node whose only
parent is the block input, then a merge node whose parents are `[block_input,
sublayer]` and which has no parameters (`thetas` entry `None`). Track the current block
input as you go.

In [ ]:
D = 4

def mk_chain(L, key):
    ks = jr.split(key, L); nodes = [([], None)]; thetas = [None]
    for i in range(L):
        nodes.append(([i], lambda p, th: jnp.tanh(th @ p[0])))
        thetas.append(jr.normal(ks[i], (D, D)) * 0.5)
    return nodes, thetas

def mk_resid_folded(L, key):
    ks = jr.split(key, L); nodes = [([], None)]; thetas = [None]
    for i in range(L):
        # the skip lives INSIDE the edge function, so the graph stays a chain
        nodes.append(([i], lambda p, th: jnp.tanh(th @ p[0]) + p[0]))
        thetas.append(jr.normal(ks[i], (D, D)) * 0.5)
    return nodes, thetas

def mk_resid_explicit(L, key):
    ks = jr.split(key, L); nodes = [([], None)]; thetas = [None]
    prev = 0                     # index of the current block's input node
    for i in range(L):
        # sublayer node: one parent (the block input), one weight
        nodes.append(([prev], lambda p, th: jnp.tanh(th @ p[0])))
        thetas.append(jr.normal(ks[i], (D, D)) * 0.5)
        a = len(nodes) - 1       # its index -- read AFTER the append, so it IS this node

        # merge node: two parents, NO parameters. This is what gives `prev` two children.
        nodes.append(([prev, a], lambda p, th: p[0] + p[1]))
        thetas.append(None)

        prev = len(nodes) - 1    # next block hangs off the MERGE, not the sublayer
    return nodes, thetas

In [ ]:
# Check
key = jr.PRNGKey(0)
xv  = jr.normal(jr.PRNGKey(1), (D,))
tgt = jr.normal(jr.PRNGKey(2), (D,))

GRAPHS = {"chain":          mk_chain(6, key),
          "resid_folded":   mk_resid_folded(6, key),
          "resid_explicit": mk_resid_explicit(3, key)}

for name, (nodes, thetas) in GRAPHS.items():
    c = children_of(nodes)
    n_multi = sum(1 for cc in c if len(cc) > 1)
    print(f"{name:16s} {len(nodes)-1} non-input nodes, "
          f"{n_multi} node(s) with >1 child")

NameError: name 'i' is not defined

**Check:**

```
chain            6 non-input nodes, 0 node(s) with >1 child
resid_folded     6 non-input nodes, 0 node(s) with >1 child
resid_explicit   6 non-input nodes, 3 node(s) with >1 child
```

Same node count in all three — deliberate, so the comparison is fair. The `0` vs `3` in
that last column is the entire difference between the two residual forms, and it is what
Section 5 will turn out to hinge on.

### The equivalence check, on all three

Same test as before, now vector-valued. Run it after any change to `infer` — this is your
regression test.

In [ ]:
def cos(a, b):
    a, b = a.ravel(), b.ravel()
    return float(jnp.dot(a, b) / (jnp.linalg.norm(a) * jnp.linalg.norm(b) + 1e-30))

for name, (nodes, thetas) in GRAPHS.items():
    tgv = true_node_grads(nodes, thetas, xv, tgt)
    v, mu, hist = infer(nodes, thetas, xv, tgt, n_steps=400, lr=0.05)
    eps  = hist[-1]
    errs = [float(jnp.max(jnp.abs(-eps[i] - tgv[i]))) for i in range(1, len(nodes))]
    coss = [cos(-eps[i], tgv[i]) for i in range(1, len(nodes))]
    print(f"{name:16s} max|-eps - dL/dv| = {max(errs):.2e}   min cosine = {min(coss):.6f}")

**Check:**

```
chain            max|-eps - dL/dv| = 1.29e-05   min cosine = 1.000000
resid_folded     max|-eps - dL/dv| = 2.57e-04   min cosine = 1.000000
resid_explicit   max|-eps - dL/dv| = 1.13e-05   min cosine = 1.000000
```

Cosine of exactly 1.0 at every node in all three graphs. The equivalence result survives
branching, skip connections, and multi-parent merge nodes — your implementation
generalises to any predictor you can write as a graph of differentiable functions.

## 5. Exercise 6 — the inference budget

On a chain you measured a staircase: layer $\ell$'s weight gradient is *exactly zero*
until inference step $L - \ell$, because the output error advances one node per step. That
gave the hard floor $T \geq \text{depth}$.

The open question for a ViT predictor is what replaces "depth" when the graph branches.

**Hypothesis to test:** the front is governed by the **shortest path from each node to the
output**, counted in edges. On a chain that equals depth. With a skip connection it
doesn't.

Two things to write. `graph_dist_to_output` computes that shortest path — walk child edges
backwards from the output, relaxing until stable (the graphs are tiny, so a naive
relaxation loop is fine). `front` measures the truth: for each node, the first step at
which $\|\epsilon_i\|_\infty$ exceeds `tol`.

**Predict first.** Does an explicit skip make the front faster, slower, or leave it
unchanged? Write your guess down before running the check.

In [ ]:
def graph_dist_to_output(nodes):
    '''Shortest number of edges from each node to the output node.'''
    ch, N = children_of(nodes), len(nodes) - 1
    INF = 10 ** 9
    d = [INF] * len(nodes); d[N] = 0
    # TODO: repeatedly relax d[i] = min over children j of (d[j] + 1) until nothing changes
    return d

def front(nodes, thetas, n_steps=14, lr=0.05, tol=1e-14):
    '''First inference step at which each node's error becomes nonzero.'''
    _, _, hist = infer(nodes, thetas, xv, tgt, n_steps=n_steps, lr=lr)
    N   = len(nodes) - 1
    out = []
    # TODO: for each node i in 1..N, scan hist for the first t where
    #       max|eps[i]| > tol, and append that t (None if never)
    return out

In [ ]:
# Check
for name, (nodes, thetas) in GRAPHS.items():
    d = graph_dist_to_output(nodes)[1:]
    f = front(nodes, thetas)
    print(f"{name:16s} graph distance to output : {d}")
    print(f"{'':16s} first-nonzero-eps step   : {f}")
    print(f"{'':16s} identical: {d == f}\n")

**Check:**

```
chain            graph distance to output : [5, 4, 3, 2, 1, 0]
                 first-nonzero-eps step   : [5, 4, 3, 2, 1, 0]
                 identical: True

resid_folded     graph distance to output : [5, 4, 3, 2, 1, 0]
                 first-nonzero-eps step   : [5, 4, 3, 2, 1, 0]
                 identical: True

resid_explicit   graph distance to output : [3, 2, 2, 1, 1, 0]
                 first-nonzero-eps step   : [3, 2, 2, 1, 1, 0]
                 identical: True
```

Exact on every node of every graph. The rule:

> A node's error stays **identically zero** until step $t$ = its shortest path to the
> output in edges. So the inference floor is not depth — it is the graph's eccentricity
> from the output node.

Now compare the two residual forms, which compute the **same function**:

- `resid_folded` — 6 nodes, front `[5,4,3,2,1,0]`, floor $T \geq 5$
- `resid_explicit` — 6 nodes, front `[3,2,2,1,1,0]`, floor $T \geq 3$

Writing the skip as an explicit merge node **shortens the longest path to the output**,
because the identity edge is a one-hop route bypassing the sublayer. Same function, fewer
sequential relaxation steps before every parameter has a gradient. For $L$ blocks written
explicitly the floor is $L+1$ rather than $2L$.

That is a genuinely useful result for your proposal: the $T \geq \text{depth}$ floor you
measured on an MLP is the *worst case*, and residual architectures are strictly better
than it. Skip connections cut the PC inference budget, not just the optimisation
difficulty.

### The floor is necessary, not sufficient

$T$ equal to the graph distance gets every node a *nonzero* gradient. It does not get it
the *correct* one. Separate the two by measuring how the equivalence error decays with
$T$. Nothing new to write — this reuses what you have.

In [ ]:
Ts = [1, 2, 3, 5, 10, 20, 50, 100, 200, 400]
eq_curves = {}
for name in ["chain", "resid_explicit"]:
    nodes, thetas = GRAPHS[name]
    tgv  = true_node_grads(nodes, thetas, xv, tgt)
    errs = []
    for T in Ts:
        _, _, hist = infer(nodes, thetas, xv, tgt, n_steps=T, lr=0.05)
        eps = hist[-1]
        errs.append(max(float(jnp.max(jnp.abs(-eps[i] - tgv[i])))
                        for i in range(1, len(nodes))))
    eq_curves[name] = errs
    print(f"{name:16s}", ["%.2e" % e for e in errs])

**Check:**

```
chain            ['1.34e+00', '1.29e+00', '1.29e+00', '1.29e+00', '1.28e+00', '1.25e+00', '9.66e-01', '4.18e-01', '2.53e-02', '1.29e-05']
resid_explicit   ['2.40e+00', '2.40e+00', '2.40e+00', '2.40e+00', '2.38e+00', '2.28e+00', '1.57e+00', '5.99e-01', '2.55e-02', '1.13e-05']
```

The gap between the two floors is stark. The front reaches every node by $T=5$ (chain) or
$T=3$ (residual), but the equivalence error at $T=5$ is still $O(1)$ — the gradients are
nonzero and wrong. Reaching $10^{-5}$ takes $T \approx 400$, roughly **80×** the
topological floor at this step size.

Two distinct budgets, and only the first is topological:

1. $T \geq$ graph distance — required for a *nonzero* gradient. Set by topology.
2. $T \approx$ hundreds — required for the gradient to *equal* backprop's. Set by the
   Euler step size $\eta_v$ and the conditioning of $\mathcal{F}$, not by the graph.

The paper's §5 cost discussion (p. 9) concerns the second budget; that's where the
100–200× serial overhead comes from.

## 6. Exercise 7 — weight updates (Equation 3)

$$\frac{d\theta_i}{dt} = -\frac{\partial \mathcal{F}}{\partial \theta_i}
= \epsilon_i \frac{\partial \hat v_i}{\partial \theta_i}$$

The reason this is local: $\theta_i$ appears in **exactly one** node's prediction — node
$i$'s — so only $\epsilon_i$ enters. No sum over children, no chain through the graph.
This is the same locality you verified numerically as a Hebbian outer product on the MLP;
here `jax.vjp` gives it for any edge function.

Two cases to handle: skip nodes whose `thetas` entry is `None` (the merge nodes have no
parameters), and evaluate the parent values at the **relaxed** `v`, not the feedforward
ones.

Return gradients with the sign convention of a loss gradient, so a plain
`theta - lr * grad` descends.

In [ ]:
def pc_weight_grads(nodes, thetas, v, mu, N):
    out = [None] * len(nodes)
    for i in range(1, N + 1):
        if thetas[i] is None:
            continue
        # TODO: eps_i = v[i] - mu[i]
        #       pv    = parent values of node i, taken from v
        #       vjp   = jax.vjp of (lambda th: fn(pv, th)) at thetas[i]
        #       out[i] = -vjp(eps_i)[0]
        
        pass
    return out

### Training: PC against backprop

Train the residual graph two ways at several values of $T$. If the equivalence result
means what it says, PC should track backprop's loss curve once $T$ is large enough, and
lag when it isn't.

**This cell is the slow one** (~2 minutes). PC at $T=200$ runs 200 sequential relaxation
steps per epoch, deliberately without `jit` so the loop stays readable.

In [ ]:
def train(mode, epochs=25, T=100, lr_w=0.05, lr_v=0.05):
    nodes, thetas = mk_resid_explicit(3, key)
    losses = []
    for ep in range(epochs):
        losses.append(float(loss_fn(nodes, thetas, xv, tgt)))
        if mode == "bp":
            wg = jax.grad(lambda th: loss_fn(nodes, th, xv, tgt))(thetas)
        else:
            # TODO: relax with infer(..., n_steps=T, lr=lr_v), then
            #       wg = pc_weight_grads(nodes, thetas, v, mu, len(nodes) - 1)
            pass
        thetas = [None if t is None else t - lr_w * wg[i] for i, t in enumerate(thetas)]
    return losses

import time
t0 = time.time(); bp_losses = train("bp"); bp_t = time.time() - t0
print(f"backprop     {bp_t:5.1f}s  final loss = {bp_losses[-1]:.6f}")

pc_losses = {}
for T in [30, 100, 200]:
    t0 = time.time(); pc_losses[T] = train("pc", T=T); dt = time.time() - t0
    gap = max(abs(a - b) for a, b in zip(bp_losses, pc_losses[T]))
    print(f"PC  T={T:<4}   {dt:5.1f}s  final loss = {pc_losses[T][-1]:.6f}   "
          f"max|bp - pc| over epochs = {gap:.3e}")

**Check** (timings vary with your machine; the losses should not):

```
backprop       3.1s  final loss = 0.284659
PC  T=30      13.0s  final loss = 1.017195   max|bp - pc| over epochs = 7.325e-01
PC  T=100     40.3s  final loss = 0.405125   max|bp - pc| over epochs = 1.246e-01
PC  T=200     73.6s  final loss = 0.285055   max|bp - pc| over epochs = 6.270e-02
```

PC converges to backprop's trajectory as $T$ grows — final loss $0.285055$ against
backprop's $0.284659$ at $T=200$ — for $24\times$ the wall-clock. At $T=30$, well above
the topological floor of 3, PC is still visibly worse ($1.017$ vs $0.285$).

That is the honest cost picture, and it matches §5 (p. 9). PC's case here is not speed and
not gradient quality; it is locality — no backward sweep, every update computable from
information at one node and its immediate neighbours.

## 7. The figure

Plotting is given complete — it isn't the learning objective, and you've already produced
every number in it.

In [ ]:
def front_matrix(nodes, thetas, n_steps=8, lr=0.05):
    _, _, hist = infer(nodes, thetas, xv, tgt, n_steps=n_steps, lr=lr)
    N = len(nodes) - 1
    return np.array([[float(jnp.max(jnp.abs(eps[i]))) for i in range(1, N + 1)]
                     for eps in hist])

M_chain = front_matrix(*GRAPHS["chain"])
M_resid = front_matrix(*GRAPHS["resid_explicit"])
print("chain: zeros in the feedforward row :", int((M_chain[0] == 0).sum()), "of", M_chain.shape[1])
print("resid: zeros in the feedforward row :", int((M_resid[0] == 0).sum()), "of", M_resid.shape[1])

In [ ]:
BASE, SMALL, TINY = 10, 9, 8
mpl.rcParams.update({
    "font.size": BASE, "axes.titlesize": BASE, "axes.labelsize": BASE,
    "legend.fontsize": SMALL, "xtick.labelsize": TINY, "ytick.labelsize": TINY,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlelocation": "left", "axes.titleweight": "normal", "figure.dpi": 110,
})
C_CHAIN, C_RESID, C_BP = "#3B6FB6", "#D1791E", "#555555"

fig = plt.figure(figsize=(11.5, 8.6))
gs  = fig.add_gridspec(2, 2, hspace=0.52, wspace=0.30)
axA, axB, axC, axD = (fig.add_subplot(gs[0,0]), fig.add_subplot(gs[0,1]),
                      fig.add_subplot(gs[1,0]), fig.add_subplot(gs[1,1]))

def panel_letter(ax, s):
    ax.text(-0.16, 1.10, s, transform=ax.transAxes,
            fontsize=BASE+3, fontweight="bold", va="top", ha="left")

# a: equivalence error vs T
axA.loglog(Ts, eq_curves["chain"],          "o-", color=C_CHAIN, lw=1.8, ms=5)
axA.loglog(Ts, eq_curves["resid_explicit"], "s-", color=C_RESID, lw=1.8, ms=5)
axA.text(1.15, eq_curves["chain"][0]*0.55,          "chain",          color=C_CHAIN, ha="left", fontsize=SMALL)
axA.text(1.15, eq_curves["resid_explicit"][0]*1.9,  "residual block", color=C_RESID, ha="left", fontsize=SMALL)
axA.set_ylim(3e-6, 1.2e1)
axA.axvline(5, color=C_CHAIN, ls=":", lw=1.2)
axA.axvline(3, color=C_RESID, ls=":", lw=1.2)
axA.annotate("topological floors\n(nonzero, still wrong)", xy=(4, 1.0),
             xytext=(11, 6e-3), fontsize=TINY, color="#444444",
             arrowprops=dict(arrowstyle="-", lw=0.8, color="#888888"))
axA.set_xlabel("inference steps $T$")
axA.set_ylabel(r"max$_i\,|-\epsilon_i - \partial L/\partial v_i|$")
axA.set_title("Errors reach the backprop gradients only\nfar beyond the topological floor")
panel_letter(axA, "a")

# b, c: propagation fronts
def draw_front(ax, M, dist, title, letter):
    Mm   = np.ma.masked_where(M <= 0, M)
    cmap = mpl.cm.viridis.copy(); cmap.set_bad("#DDDDDD")
    im   = ax.imshow(np.log10(Mm), cmap=cmap, aspect="auto", origin="upper")
    for t in range(M.shape[0]):
        for i in range(M.shape[1]):
            if M[t, i] <= 0:
                ax.text(i, t, "0", ha="center", va="center", fontsize=TINY, color="#666666")
    ax.set_xticks(range(M.shape[1]))
    ax.set_xticklabels([f"$v_{{{i+1}}}$" for i in range(M.shape[1])])
    ax.set_yticks(range(M.shape[0]))
    ax.set_ylabel("inference step $t$"); ax.set_xlabel("node")
    ax.set_title(title)
    cb = plt.colorbar(im, ax=ax, pad=0.03, fraction=0.046)
    cb.ax.tick_params(labelsize=TINY)
    cb.ax.set_title(r"$\log_{10}\|\epsilon_i\|_\infty$", fontsize=TINY, pad=6, loc="left")
    ax.plot(range(M.shape[1]), dist, "o-", color="#C0392B", lw=1.6, ms=4.5,
            label="shortest path to output")
    ax.legend(frameon=False, fontsize=TINY, loc="lower left")
    panel_letter(ax, letter)

draw_front(axB, M_chain, graph_dist_to_output(GRAPHS["chain"][0])[1:],
           "Chain: front advances one node per step", "b")
draw_front(axC, M_resid, graph_dist_to_output(GRAPHS["resid_explicit"][0])[1:],
           "Residual block: skip edges shorten the front", "c")

# d: training
axD.semilogy(bp_losses, color=C_BP, lw=2.4, label="backprop")
for T, sty in zip([30, 100, 200], [":", "--", "-"]):
    axD.semilogy(pc_losses[T], sty, color=C_RESID, lw=1.7, label=f"PC, $T$={T}")
axD.set_xlabel("epoch"); axD.set_ylabel("loss")
axD.set_title("PC reaches backprop's trajectory\nonly at large $T$")
axD.set_yticks([0.3, 0.5, 1.0, 1.5])
axD.set_yticklabels(["0.3", "0.5", "1.0", "1.5"])
axD.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())
axD.legend(frameon=False, loc="lower left")
axD.margins(0.04)
panel_letter(axD, "d")

fig.suptitle("Predictive coding on branching graphs: topology sets the inference floor, "
             "step count sets the accuracy", fontsize=BASE+1, y=1.00)
fig.savefig("graph_pc_results.png", dpi=200, bbox_inches="tight")
print("saved graph_pc_results.png")

## 8. Where this leaves your proposal

**What you now have.** Equation 2 for an arbitrary DAG, checked against autodiff to
cosine 1.0 on a branch point, a folded skip, and an explicit residual merge. Equation 3
for arbitrary edge functions. This is the piece the chain implementation was missing, and
it generalises to any predictor expressible as a graph of differentiable functions —
attention included, once decomposed into nodes.

**Two measured results to carry forward.**

1. The gradient-availability floor is the graph's distance to the output, not its depth.
   Explicit residual connections give $L+1$ rather than $2L$ for $L$ blocks.
2. That floor is necessary and nowhere near sufficient. Matching backprop to $10^{-5}$
   took $T \approx 400$ here (~80× the floor), and in training $T=200$ cost $24\times$
   the wall-clock for a loss matching backprop to 3 decimals.

**The tension your proposal has to resolve.** This paper proves PC's errors converge to
backprop's gradients. At convergence, then, PC's gradient cannot be *better* than
backprop's — it is the same vector. So "better energy gradient than GD" needs sharpening.
The candidates, ranked:

- **Learnable precisions** ($\Sigma_i^{-1} \neq I$) — the one place the paper says
  something genuinely new is available (§5, p. 10). Keeping the precisions gives an
  uncertainty-weighted objective backprop does not compute: a *different* objective, not a
  different route to the same one, which is what "better" requires. Untested in the paper.
- **Locality as an implementation argument** — parallel updates, no backward sweep. Real,
  but it's a hardware claim, and the serial $T$ cost above works against it on a GPU.
- **Truncated inference** ($T$ below convergence) — you already measured that this mainly
  shrinks gradient magnitude rather than changing direction, so it behaves like a smaller
  learning rate. Weakest of the three.

**Natural next exercises in this notebook.**

1. Add attention as a graph: three projection nodes plus a multiplicative merge. That
   merge has three parents, which your `infer` already handles — verify with the
   equivalence check rather than assuming.
2. Add per-node precisions to `infer`, with the step size scaled inversely to precision
   (you established why: precision enters the discretised update as stiffness, so one
   global $\eta_v$ won't do). Then test whether the uncertainty-weighted objective does
   anything plain MSE doesn't.